# Model Building (Updated for `light_curves.csv`)
**Purpose:** Train a self-supervised contrastive learning encoder (SimCLR-style) on multi-band light-curve sequences.

**Inputs (preferred):**
- `outputs/light_curve_pairs.npz` from `Data_Augmentation_updated_for_light_curves.ipynb`
- or `outputs/light_curves_sequences.npz` from `Preprocessing_updated_for_light_curves.ipynb`

**Outputs:**
- Trained encoder + projection head checkpoints
- Validation embeddings + clustering metrics (silhouette)


In [1]:
import os
import pickle
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler


# =========================
# 1) Config
# =========================
CSV_PATH = "outputs/selected_features_full.csv"   # change if needed
OUT_PKL  = "simclr_model.pkl"

BATCH_SIZE = 512
EPOCHS = 30
LR = 1e-3
TEMPERATURE = 0.2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Pick the columns you want to train on (simple + explicit)
FEATURE_COLS = ["mjd", "fid", "magpsf", "sigmapsf", "ra", "dec", "isdiffpos"]


# =========================
# 2) Dataset + augmentations
# =========================
class SimCLRTabularDataset(Dataset):
    def __init__(self, X: np.ndarray, noise_std: float = 0.02, drop_prob: float = 0.10):
        """
        X: standardized numpy array (N, D)
        noise_std: gaussian noise strength (on standardized features)
        drop_prob: probability of masking a feature to 0 (feature dropout)
        """
        self.X = X.astype(np.float32)
        self.noise_std = noise_std
        self.drop_prob = drop_prob

    def _augment(self, x: torch.Tensor) -> torch.Tensor:
        # 1) Gaussian noise
        x = x + torch.randn_like(x) * self.noise_std

        # 2) Feature dropout (randomly mask some features)
        if self.drop_prob > 0:
            mask = (torch.rand_like(x) > self.drop_prob).float()
            x = x * mask

        return x

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx])
        x1 = self._augment(x.clone())
        x2 = self._augment(x.clone())
        return x1, x2


# =========================
# 3) SimCLR Model (MLP encoder + projection head)
# =========================
class MLPEncoder(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 256, emb_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, emb_dim),
        )

    def forward(self, x):
        return self.net(x)


class ProjectionHead(nn.Module):
    def __init__(self, in_dim: int, proj_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(),
            nn.Linear(in_dim, proj_dim),
        )

    def forward(self, x):
        return self.net(x)


class SimCLR(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 256, emb_dim: int = 128, proj_dim: int = 128):
        super().__init__()
        self.encoder = MLPEncoder(in_dim, hidden_dim=hidden_dim, emb_dim=emb_dim)
        self.projector = ProjectionHead(emb_dim, proj_dim=proj_dim)

    def forward(self, x):
        h = self.encoder(x)              # (B, emb_dim)
        z = self.projector(h)            # (B, proj_dim)
        z = F.normalize(z, dim=1)
        return h, z


# =========================
# 4) NT-Xent loss
# =========================
def nt_xent_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.2) -> torch.Tensor:
    """
    z1, z2: normalized (B, D)
    """
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)  # (2B, D)

    # cosine similarity matrix
    sim = torch.mm(z, z.t()) / temperature  # (2B, 2B)

    # mask self-similarity
    mask = torch.eye(2 * B, device=z.device).bool()
    sim = sim.masked_fill(mask, -1e9)

    # positives: (i, i+B) and (i+B, i)
    pos = torch.cat([torch.diag(sim, B), torch.diag(sim, -B)], dim=0)  # (2B,)

    # denominator: logsumexp over all except itself
    loss = -pos + torch.logsumexp(sim, dim=1)
    return loss.mean()


# =========================
# 5) Train
# =========================
def main():
    # Load
    df = pd.read_csv(CSV_PATH)

    # Keep only required feature columns that exist
    cols = [c for c in FEATURE_COLS if c in df.columns]
    if len(cols) < 2:
        raise ValueError(f"Not enough usable feature columns found. Found: {cols}")

    Xdf = df[cols].copy()

    # Convert to numeric + clean
    for c in cols:
        Xdf[c] = pd.to_numeric(Xdf[c], errors="coerce")
    Xdf = Xdf.replace([np.inf, -np.inf], np.nan)

    # Fill NaNs with median
    Xdf = Xdf.fillna(Xdf.median(numeric_only=True))

    # Standardize
    scaler = StandardScaler()
    X = scaler.fit_transform(Xdf.values)

    # DataLoader
    ds = SimCLRTabularDataset(X, noise_std=0.03, drop_prob=0.10)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)

    # Model
    model = SimCLR(in_dim=X.shape[1], hidden_dim=256, emb_dim=128, proj_dim=128).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    model.train()
    for epoch in range(1, EPOCHS + 1):
        losses = []
        for x1, x2 in dl:
            x1 = x1.to(DEVICE)
            x2 = x2.to(DEVICE)

            _, z1 = model(x1)
            _, z2 = model(x2)

            loss = nt_xent_loss(z1, z2, temperature=TEMPERATURE)

            opt.zero_grad()
            loss.backward()
            opt.step()

            losses.append(loss.item())

        print(f"Epoch {epoch:02d}/{EPOCHS} | loss={np.mean(losses):.4f}")

    # =========================
    # 6) Save as PKL
    # =========================
    package = {
        "feature_cols": cols,
        "scaler": scaler,
        "model_state_dict": model.state_dict(),
        "model_config": {
            "in_dim": X.shape[1],
            "hidden_dim": 256,
            "emb_dim": 128,
            "proj_dim": 128
        }
    }

    with open(OUT_PKL, "wb") as f:
        pickle.dump(package, f)

    print(f"\n✅ Saved SimCLR package to: {OUT_PKL}")


if __name__ == "__main__":
    main()


Epoch 01/30 | loss=4.6669
Epoch 02/30 | loss=4.0965
Epoch 03/30 | loss=3.8928
Epoch 04/30 | loss=3.7769
Epoch 05/30 | loss=3.6611
Epoch 06/30 | loss=3.5972
Epoch 07/30 | loss=3.5359
Epoch 08/30 | loss=3.4849
Epoch 09/30 | loss=3.4653
Epoch 10/30 | loss=3.4448
Epoch 11/30 | loss=3.4220
Epoch 12/30 | loss=3.3943
Epoch 13/30 | loss=3.3886
Epoch 14/30 | loss=3.3654
Epoch 15/30 | loss=3.3646
Epoch 16/30 | loss=3.3436
Epoch 17/30 | loss=3.3210
Epoch 18/30 | loss=3.3168
Epoch 19/30 | loss=3.3161
Epoch 20/30 | loss=3.3005
Epoch 21/30 | loss=3.2945
Epoch 22/30 | loss=3.2823
Epoch 23/30 | loss=3.2888
Epoch 24/30 | loss=3.2831
Epoch 25/30 | loss=3.2694
Epoch 26/30 | loss=3.2520
Epoch 27/30 | loss=3.2565
Epoch 28/30 | loss=3.2558
Epoch 29/30 | loss=3.2504
Epoch 30/30 | loss=3.2424

✅ Saved SimCLR package to: simclr_model.pkl


In [5]:
"""
Enhanced + Corrected SimCLR (Tabular) for ZTF selected_features_full.csv

Fixes + upgrades vs your version:
✅ Drop exact-duplicate rows (important for contrastive learning)
✅ Proper handling of categorical columns (fid one-hot, isdiffpos -> 0/1)
✅ Scale ONLY continuous columns (not categorical)
✅ Stronger but safer augmentations (noise, feature dropout on continuous only, scale jitter, mixup)
✅ Optional evaluation metrics (kNN / linear probe / retrieval) IF you have labels
✅ Collapse check (embedding std)
✅ Early stopping (based on kNN acc if labels exist, else loss)

Outputs:
- simclr_model_enhanced.pkl (scaler + encoder weights + configs)
"""

import os
import pickle
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression


# =========================
# 1) Config
# =========================
CSV_PATH = "./outputs/selected_features_full.csv"   # <-- your uploaded file path
OUT_PKL  = "simclr_model_enhanced.pkl"

# If you have labels, set this to your column name. Otherwise keep None.
LABEL_COL = None   # e.g., "label" or "transient_type"

# Base features available in your CSV
FEATURE_COLS = ["mjd", "fid", "magpsf", "sigmapsf", "ra", "dec", "isdiffpos"]

BATCH_SIZE = 512
EPOCHS = 150
LR = 1e-3
WEIGHT_DECAY = 1e-4
TEMPERATURE = 0.2

VAL_SPLIT = 0.15
SEED = 42

# Augmentation strengths (on standardized continuous features)
NOISE_STD = 0.05
DROP_PROB = 0.10
SCALE_JITTER = 0.05
MIXUP_ALPHA = 0.2

# Evaluation (only if labels exist)
KNN_K = 20
EARLY_STOP_PATIENCE = 10

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = (DEVICE == "cuda")


# =========================
# 2) Dataset + augmentations
# =========================
class SimCLRTabularDataset(Dataset):
    def __init__(self, X: np.ndarray):
        self.X = X.astype(np.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx])


def augment_batch(
    xb: torch.Tensor,
    cont_dim: int,
    noise_std=0.05,
    drop_prob=0.10,
    scale_jitter=0.05,
    mixup_alpha=0.2,
):
    """
    xb: (B, D) where first cont_dim features are continuous (standardized)
        remaining features are categorical/boolean (0/1)
    We apply strong augmentations ONLY to continuous part to avoid corrupting categories.
    """
    x = xb.clone()
    xc = x[:, :cont_dim]
    xe = x[:, cont_dim:]  # categorical/boolean part

    # 1) Gaussian noise on continuous
    xc = xc + torch.randn_like(xc) * noise_std

    # 2) Feature dropout on continuous
    if drop_prob > 0:
        mask = (torch.rand_like(xc) > drop_prob).float()
        xc = xc * mask

    # 3) Multiplicative jitter on continuous
    if scale_jitter > 0:
        xc = xc * (1.0 + torch.randn_like(xc) * scale_jitter)

    # 4) Mixup-style within batch (continuous only)
    if mixup_alpha and mixup_alpha > 0:
        lam = torch.rand((xc.size(0), 1), device=xc.device) * mixup_alpha
        xc_roll = torch.roll(xc, shifts=1, dims=0)
        xc = lam * xc + (1 - lam) * xc_roll

    # Recombine (keep categories unchanged)
    x_aug = torch.cat([xc, xe], dim=1)
    return x_aug


# =========================
# 3) SimCLR Model
# =========================
class MLPEncoder(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 256, emb_dim: int = 128, p_drop: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(p_drop),

            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(p_drop),

            nn.Linear(hidden_dim, emb_dim),
        )

    def forward(self, x):
        return self.net(x)


class ProjectionHead(nn.Module):
    def __init__(self, in_dim: int, proj_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(),
            nn.Linear(in_dim, proj_dim),
        )

    def forward(self, x):
        return self.net(x)


class SimCLR(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 256, emb_dim: int = 128, proj_dim: int = 128):
        super().__init__()
        self.encoder = MLPEncoder(in_dim, hidden_dim=hidden_dim, emb_dim=emb_dim, p_drop=0.1)
        self.projector = ProjectionHead(emb_dim, proj_dim=proj_dim)

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        z = F.normalize(z, dim=1)
        return h, z


# =========================
# 4) NT-Xent Loss
# =========================
def nt_xent_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.2) -> torch.Tensor:
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)               # (2B, D)
    sim = torch.mm(z, z.t()) / temperature       # (2B, 2B)

    # mask self similarity
    mask = torch.eye(2 * B, device=z.device).bool()
    sim = sim.masked_fill(mask, -1e9)

    # positives
    pos = torch.cat([torch.diag(sim, B), torch.diag(sim, -B)], dim=0)  # (2B,)
    loss = -pos + torch.logsumexp(sim, dim=1)
    return loss.mean()


# =========================
# 5) Optional Evaluation Metrics (requires labels)
# =========================
@torch.no_grad()
def encode_all(encoder: nn.Module, X: np.ndarray, batch_size=2048, device="cpu"):
    encoder.eval()
    feats = []
    dl = DataLoader(torch.from_numpy(X.astype(np.float32)), batch_size=batch_size, shuffle=False)
    for xb in dl:
        xb = xb.to(device)
        h = encoder(xb)
        h = F.normalize(h, dim=1)
        feats.append(h.cpu().numpy())
    return np.vstack(feats)


def knn_accuracy(train_emb, train_y, val_emb, val_y, k=20):
    sim = val_emb @ train_emb.T
    topk = np.argpartition(-sim, kth=min(k, sim.shape[1]-1), axis=1)[:, :k]

    preds = []
    for i in range(topk.shape[0]):
        neigh = train_y[topk[i]]
        vals, cnts = np.unique(neigh, return_counts=True)
        preds.append(vals[np.argmax(cnts)])
    preds = np.array(preds)

    return (
        accuracy_score(val_y, preds),
        f1_score(val_y, preds, average="macro")
    )


def retrieval_recall_at_k(val_emb, val_y, k_list=(1, 5)):
    sim = val_emb @ val_emb.T
    np.fill_diagonal(sim, -1e9)
    order = np.argsort(-sim, axis=1)

    out = {}
    for k in k_list:
        hit = 0
        for i in range(val_emb.shape[0]):
            nn_idx = order[i, :k]
            if np.any(val_y[nn_idx] == val_y[i]):
                hit += 1
        out[f"R@{k}"] = hit / val_emb.shape[0]
    return out


def linear_probe(train_emb, train_y, val_emb, val_y):
    clf = LogisticRegression(max_iter=2000, n_jobs=-1)
    clf.fit(train_emb, train_y)
    pred = clf.predict(val_emb)
    return (
        accuracy_score(val_y, pred),
        f1_score(val_y, pred, average="macro")
    )


# =========================
# 6) Main
# =========================
def main():
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    # ---- Load + Deduplicate (CRITICAL for SimCLR) ----
    df = pd.read_csv(CSV_PATH)
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    after = len(df)
    print(f"Deduplicated rows: {before} -> {after}  (removed {before-after})")

    # ---- Ensure required columns exist ----
    cols = [c for c in FEATURE_COLS if c in df.columns]
    if len(cols) < 2:
        raise ValueError(f"Not enough usable feature columns. Found: {cols}")

    # ---- Labels (optional) ----
    has_labels = (LABEL_COL is not None and LABEL_COL in df.columns)
    if LABEL_COL is not None and not has_labels:
        print(f"⚠️ LABEL_COL='{LABEL_COL}' not found. Training without accuracy metrics.")

    y = df[LABEL_COL].values if has_labels else None

    # ---- Clean numeric ----
    Xdf = df[cols].copy()
    for c in cols:
        Xdf[c] = pd.to_numeric(Xdf[c], errors="coerce")
    Xdf = Xdf.replace([np.inf, -np.inf], np.nan)

    # ---- Handle categorical properly ----
    # fid: one-hot; isdiffpos: -1/1 -> 0/1
    if "fid" in Xdf.columns:
        Xdf["fid"] = Xdf["fid"].fillna(0).astype(int)
    if "isdiffpos" in Xdf.columns:
        Xdf["isdiffpos"] = Xdf["isdiffpos"].fillna(0)
        Xdf["isdiffpos"] = (Xdf["isdiffpos"] > 0).astype(int)

    # Fill remaining NaNs (continuous) with median
    Xdf = Xdf.fillna(Xdf.median(numeric_only=True))

    # Split continuous vs categorical
    cont_cols = [c for c in ["mjd", "magpsf", "sigmapsf", "ra", "dec"] if c in Xdf.columns]
    cat_cols = []
    if "fid" in Xdf.columns:
        # one-hot fid
        fid_oh = pd.get_dummies(Xdf["fid"], prefix="fid")
        Xdf = pd.concat([Xdf.drop(columns=["fid"]), fid_oh], axis=1)
        cat_cols += list(fid_oh.columns)
    if "isdiffpos" in Xdf.columns:
        cat_cols += ["isdiffpos"]

    # Optional: reduce shortcut leakage (try this if needed)
    # If you suspect dec is too dominant, uncomment:
    # if "dec" in cont_cols:
    #     cont_cols.remove("dec")

    # Build arrays
    X_cont = Xdf[cont_cols].values.astype(np.float32)
    X_cat  = Xdf[cat_cols].values.astype(np.float32) if len(cat_cols) else None

    # Train/val split (stratify if labels exist)
    if has_labels:
        Xc_tr, Xc_va, y_tr, y_va = train_test_split(
            X_cont, y, test_size=VAL_SPLIT, random_state=SEED, stratify=y
        )
        if X_cat is not None:
            Xk_tr, Xk_va = train_test_split(
                X_cat, test_size=VAL_SPLIT, random_state=SEED, stratify=y
            )
        else:
            Xk_tr = Xk_va = None
    else:
        Xc_tr, Xc_va = train_test_split(
            X_cont, test_size=VAL_SPLIT, random_state=SEED
        )
        if X_cat is not None:
            Xk_tr, Xk_va = train_test_split(
                X_cat, test_size=VAL_SPLIT, random_state=SEED
            )
        else:
            Xk_tr = Xk_va = None
        y_tr = y_va = None

    # Standardize continuous only (fit train)
    scaler = StandardScaler()
    Xc_tr = scaler.fit_transform(Xc_tr).astype(np.float32)
    Xc_va = scaler.transform(Xc_va).astype(np.float32)

    # Combine cont + cat
    if Xk_tr is not None:
        X_train = np.concatenate([Xc_tr, Xk_tr], axis=1).astype(np.float32)
        X_val   = np.concatenate([Xc_va, Xk_va], axis=1).astype(np.float32)
    else:
        X_train = Xc_tr.astype(np.float32)
        X_val   = Xc_va.astype(np.float32)

    cont_dim = Xc_tr.shape[1]  # first part to augment

    # Dataloader
    ds_train = SimCLRTabularDataset(X_train)
    dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)

    # Model
    model = SimCLR(in_dim=X_train.shape[1], hidden_dim=256, emb_dim=128, proj_dim=128).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    amp_scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_score = -1.0
    best_state = None
    no_improve = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []

        for xb in dl_train:
            xb = xb.to(DEVICE)

            x1 = augment_batch(xb, cont_dim, NOISE_STD, DROP_PROB, SCALE_JITTER, MIXUP_ALPHA)
            x2 = augment_batch(xb, cont_dim, NOISE_STD, DROP_PROB, SCALE_JITTER, MIXUP_ALPHA)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                _, z1 = model(x1)
                _, z2 = model(x2)
                loss = nt_xent_loss(z1, z2, temperature=TEMPERATURE)

            opt.zero_grad(set_to_none=True)
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            amp_scaler.step(opt)
            amp_scaler.update()

            losses.append(loss.item())

        # Collapse check (embedding std)
        model.eval()
        with torch.no_grad():
            xb_chk = next(iter(dl_train)).to(DEVICE)
            _, z_chk = model(xb_chk)
            emb_std = z_chk.std(dim=0).mean().item()

        msg = f"Epoch {epoch:03d}/{EPOCHS} | loss={np.mean(losses):.4f} | emb_std={emb_std:.4f}"

        # Optional: evaluation metrics
        if has_labels:
            train_emb = encode_all(model.encoder, X_train, device=DEVICE)
            val_emb   = encode_all(model.encoder, X_val, device=DEVICE)

            knn_acc, knn_f1 = knn_accuracy(train_emb, y_tr, val_emb, y_va, k=KNN_K)
            lp_acc, lp_f1   = linear_probe(train_emb, y_tr, val_emb, y_va)
            recalls         = retrieval_recall_at_k(val_emb, y_va, k_list=(1, 5))

            msg += (
                f" | kNN@{KNN_K} acc={knn_acc:.3f} f1={knn_f1:.3f}"
                f" | LP acc={lp_acc:.3f} f1={lp_f1:.3f}"
                f" | R@1={recalls['R@1']:.3f} R@5={recalls['R@5']:.3f}"
            )

            # Early stopping on knn acc
            score = knn_acc
            if score > best_score + 1e-4:
                best_score = score
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= EARLY_STOP_PATIENCE:
                    print(msg)
                    print(f"⏹️ Early stopping: no improvement for {EARLY_STOP_PATIENCE} epochs.")
                    break
        else:
            # Early stop on loss if no labels
            score = -np.mean(losses)
            if score > best_score + 1e-4:
                best_score = score
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= EARLY_STOP_PATIENCE:
                    print(msg)
                    print(f"⏹️ Early stopping: loss not improving for {EARLY_STOP_PATIENCE} epochs.")
                    break

        print(msg)

    # Restore best
    if best_state is not None:
        model.load_state_dict(best_state)
        print("✅ Restored best model weights.")

    # Save package
    package = {
        "feature_cols_original": cols,
        "cont_cols": cont_cols,
        "cat_cols": cat_cols,
        "label_col": LABEL_COL if has_labels else None,
        "scaler": scaler,
        "model_state_dict": model.state_dict(),
        "model_config": {
            "in_dim": X_train.shape[1],
            "hidden_dim": 256,
            "emb_dim": 128,
            "proj_dim": 128,
        },
        "train_config": {
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "temperature": TEMPERATURE,
            "val_split": VAL_SPLIT,
            "seed": SEED,
            "noise_std": NOISE_STD,
            "drop_prob": DROP_PROB,
            "scale_jitter": SCALE_JITTER,
            "mixup_alpha": MIXUP_ALPHA,
            "knn_k": KNN_K,
        },
    }

    with open(OUT_PKL, "wb") as f:
        pickle.dump(package, f)

    print(f"\n✅ Saved enhanced SimCLR package to: {OUT_PKL}")


if __name__ == "__main__":
    main()


Deduplicated rows: 20000 -> 15372  (removed 4628)


C:\Users\NIPUN\AppData\Local\Temp\ipykernel_21732\805319058.py:357: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp_scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
C:\Users\NIPUN\AppData\Local\Temp\ipykernel_21732\805319058.py:373: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Epoch 001/150 | loss=4.2814 | emb_std=0.0853
Epoch 002/150 | loss=3.8392 | emb_std=0.0849
Epoch 003/150 | loss=3.7530 | emb_std=0.0853
Epoch 004/150 | loss=3.7271 | emb_std=0.0855
Epoch 005/150 | loss=3.6886 | emb_std=0.0852
Epoch 006/150 | loss=3.6590 | emb_std=0.0855
Epoch 007/150 | loss=3.6249 | emb_std=0.0857
Epoch 008/150 | loss=3.6231 | emb_std=0.0854
Epoch 009/150 | loss=3.5954 | emb_std=0.0855
Epoch 010/150 | loss=3.5852 | emb_std=0.0856
⏹️ Early stopping: loss not improving for 10 epochs.

✅ Saved enhanced SimCLR package to: simclr_model_enhanced.pkl


In [6]:
"""
Enhanced + Corrected SimCLR (Tabular) for ZTF selected_features_full.csv

Fixes + upgrades vs your version:
✅ Drop exact-duplicate rows (important for contrastive learning)
✅ Proper handling of categorical columns (fid one-hot, isdiffpos -> 0/1)
✅ Scale ONLY continuous columns (not categorical)
✅ Stronger but safer augmentations (noise, feature dropout on continuous only, scale jitter, mixup)
✅ Optional evaluation metrics (kNN / linear probe / retrieval) IF you have labels
✅ Collapse check (embedding std)

Outputs:
- simclr_model_enhanced.pkl (scaler + encoder weights + configs)
"""

import os
import pickle
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression


# =========================
# 1) Config
# =========================
CSV_PATH = "./outputs/selected_features_full.csv"
OUT_PKL  = "simclr_model_enhanced.pkl"

# If you have labels, set this to your column name. Otherwise keep None.
LABEL_COL = None   # e.g., "label" or "transient_type"

FEATURE_COLS = ["mjd", "fid", "magpsf", "sigmapsf", "ra", "dec", "isdiffpos"]

BATCH_SIZE = 512
EPOCHS = 150
LR = 1e-3
WEIGHT_DECAY = 1e-4
TEMPERATURE = 0.2

VAL_SPLIT = 0.15
SEED = 42

NOISE_STD = 0.05
DROP_PROB = 0.10
SCALE_JITTER = 0.05
MIXUP_ALPHA = 0.2

KNN_K = 20

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = (DEVICE == "cuda")


# =========================
# 2) Dataset + augmentations
# =========================
class SimCLRTabularDataset(Dataset):
    def __init__(self, X: np.ndarray):
        self.X = X.astype(np.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx])


def augment_batch(
    xb: torch.Tensor,
    cont_dim: int,
    noise_std=0.05,
    drop_prob=0.10,
    scale_jitter=0.05,
    mixup_alpha=0.2,
):
    x = xb.clone()
    xc = x[:, :cont_dim]
    xe = x[:, cont_dim:]

    xc = xc + torch.randn_like(xc) * noise_std

    if drop_prob > 0:
        mask = (torch.rand_like(xc) > drop_prob).float()
        xc = xc * mask

    if scale_jitter > 0:
        xc = xc * (1.0 + torch.randn_like(xc) * scale_jitter)

    if mixup_alpha and mixup_alpha > 0:
        lam = torch.rand((xc.size(0), 1), device=xc.device) * mixup_alpha
        xc_roll = torch.roll(xc, shifts=1, dims=0)
        xc = lam * xc + (1 - lam) * xc_roll

    return torch.cat([xc, xe], dim=1)


# =========================
# 3) SimCLR Model
# =========================
class MLPEncoder(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 256, emb_dim: int = 128, p_drop: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(p_drop),

            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(p_drop),

            nn.Linear(hidden_dim, emb_dim),
        )

    def forward(self, x):
        return self.net(x)


class ProjectionHead(nn.Module):
    def __init__(self, in_dim: int, proj_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(),
            nn.Linear(in_dim, proj_dim),
        )

    def forward(self, x):
        return self.net(x)


class SimCLR(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 256, emb_dim: int = 128, proj_dim: int = 128):
        super().__init__()
        self.encoder = MLPEncoder(in_dim, hidden_dim=hidden_dim, emb_dim=emb_dim, p_drop=0.1)
        self.projector = ProjectionHead(emb_dim, proj_dim=proj_dim)

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        z = F.normalize(z, dim=1)
        return h, z


# =========================
# 4) NT-Xent Loss
# =========================
def nt_xent_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.2) -> torch.Tensor:
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = torch.mm(z, z.t()) / temperature

    mask = torch.eye(2 * B, device=z.device).bool()
    sim = sim.masked_fill(mask, -1e9)

    pos = torch.cat([torch.diag(sim, B), torch.diag(sim, -B)], dim=0)
    loss = -pos + torch.logsumexp(sim, dim=1)
    return loss.mean()


# =========================
# 5) Optional evaluation (requires labels)
# =========================
@torch.no_grad()
def encode_all(encoder: nn.Module, X: np.ndarray, batch_size=2048, device="cpu"):
    encoder.eval()
    feats = []
    dl = DataLoader(torch.from_numpy(X.astype(np.float32)), batch_size=batch_size, shuffle=False)
    for xb in dl:
        xb = xb.to(device)
        h = encoder(xb)
        h = F.normalize(h, dim=1)
        feats.append(h.cpu().numpy())
    return np.vstack(feats)


def knn_accuracy(train_emb, train_y, val_emb, val_y, k=20):
    sim = val_emb @ train_emb.T
    topk = np.argpartition(-sim, kth=min(k, sim.shape[1]-1), axis=1)[:, :k]

    preds = []
    for i in range(topk.shape[0]):
        neigh = train_y[topk[i]]
        vals, cnts = np.unique(neigh, return_counts=True)
        preds.append(vals[np.argmax(cnts)])
    preds = np.array(preds)

    return (
        accuracy_score(val_y, preds),
        f1_score(val_y, preds, average="macro")
    )


def retrieval_recall_at_k(val_emb, val_y, k_list=(1, 5)):
    sim = val_emb @ val_emb.T
    np.fill_diagonal(sim, -1e9)
    order = np.argsort(-sim, axis=1)

    out = {}
    for k in k_list:
        hit = 0
        for i in range(val_emb.shape[0]):
            nn_idx = order[i, :k]
            if np.any(val_y[nn_idx] == val_y[i]):
                hit += 1
        out[f"R@{k}"] = hit / val_emb.shape[0]
    return out


def linear_probe(train_emb, train_y, val_emb, val_y):
    clf = LogisticRegression(max_iter=2000, n_jobs=-1)
    clf.fit(train_emb, train_y)
    pred = clf.predict(val_emb)
    return (
        accuracy_score(val_y, pred),
        f1_score(val_y, pred, average="macro")
    )


# =========================
# 6) Main
# =========================
def main():
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    df = pd.read_csv(CSV_PATH)
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    after = len(df)
    print(f"Deduplicated rows: {before} -> {after}  (removed {before-after})")

    cols = [c for c in FEATURE_COLS if c in df.columns]
    if len(cols) < 2:
        raise ValueError(f"Not enough usable feature columns. Found: {cols}")

    has_labels = (LABEL_COL is not None and LABEL_COL in df.columns)
    if LABEL_COL is not None and not has_labels:
        print(f"⚠️ LABEL_COL='{LABEL_COL}' not found. Training without accuracy metrics.")

    y = df[LABEL_COL].values if has_labels else None

    Xdf = df[cols].copy()
    for c in cols:
        Xdf[c] = pd.to_numeric(Xdf[c], errors="coerce")
    Xdf = Xdf.replace([np.inf, -np.inf], np.nan)

    if "fid" in Xdf.columns:
        Xdf["fid"] = Xdf["fid"].fillna(0).astype(int)
    if "isdiffpos" in Xdf.columns:
        Xdf["isdiffpos"] = Xdf["isdiffpos"].fillna(0)
        Xdf["isdiffpos"] = (Xdf["isdiffpos"] > 0).astype(int)

    Xdf = Xdf.fillna(Xdf.median(numeric_only=True))

    cont_cols = [c for c in ["mjd", "magpsf", "sigmapsf", "ra", "dec"] if c in Xdf.columns]
    cat_cols = []

    if "fid" in Xdf.columns:
        fid_oh = pd.get_dummies(Xdf["fid"], prefix="fid")
        Xdf = pd.concat([Xdf.drop(columns=["fid"]), fid_oh], axis=1)
        cat_cols += list(fid_oh.columns)
    if "isdiffpos" in Xdf.columns:
        cat_cols += ["isdiffpos"]

    X_cont = Xdf[cont_cols].values.astype(np.float32)
    X_cat  = Xdf[cat_cols].values.astype(np.float32) if len(cat_cols) else None

    if has_labels:
        Xc_tr, Xc_va, y_tr, y_va = train_test_split(
            X_cont, y, test_size=VAL_SPLIT, random_state=SEED, stratify=y
        )
        if X_cat is not None:
            Xk_tr, Xk_va = train_test_split(
                X_cat, test_size=VAL_SPLIT, random_state=SEED, stratify=y
            )
        else:
            Xk_tr = Xk_va = None
    else:
        Xc_tr, Xc_va = train_test_split(X_cont, test_size=VAL_SPLIT, random_state=SEED)
        if X_cat is not None:
            Xk_tr, Xk_va = train_test_split(X_cat, test_size=VAL_SPLIT, random_state=SEED)
        else:
            Xk_tr = Xk_va = None
        y_tr = y_va = None

    scaler = StandardScaler()
    Xc_tr = scaler.fit_transform(Xc_tr).astype(np.float32)
    Xc_va = scaler.transform(Xc_va).astype(np.float32)

    if Xk_tr is not None:
        X_train = np.concatenate([Xc_tr, Xk_tr], axis=1).astype(np.float32)
        X_val   = np.concatenate([Xc_va, Xk_va], axis=1).astype(np.float32)
    else:
        X_train = Xc_tr.astype(np.float32)
        X_val   = Xc_va.astype(np.float32)

    cont_dim = Xc_tr.shape[1]

    ds_train = SimCLRTabularDataset(X_train)
    dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)

    model = SimCLR(in_dim=X_train.shape[1], hidden_dim=256, emb_dim=128, proj_dim=128).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    amp_scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []

        for xb in dl_train:
            xb = xb.to(DEVICE)

            x1 = augment_batch(xb, cont_dim, NOISE_STD, DROP_PROB, SCALE_JITTER, MIXUP_ALPHA)
            x2 = augment_batch(xb, cont_dim, NOISE_STD, DROP_PROB, SCALE_JITTER, MIXUP_ALPHA)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                _, z1 = model(x1)
                _, z2 = model(x2)
                loss = nt_xent_loss(z1, z2, temperature=TEMPERATURE)

            opt.zero_grad(set_to_none=True)
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            amp_scaler.step(opt)
            amp_scaler.update()

            losses.append(loss.item())

        # Collapse check
        model.eval()
        with torch.no_grad():
            xb_chk = next(iter(dl_train)).to(DEVICE)
            _, z_chk = model(xb_chk)
            emb_std = z_chk.std(dim=0).mean().item()

        msg = f"Epoch {epoch:03d}/{EPOCHS} | loss={np.mean(losses):.4f} | emb_std={emb_std:.4f}"

        # Optional evaluation
        if has_labels:
            train_emb = encode_all(model.encoder, X_train, device=DEVICE)
            val_emb   = encode_all(model.encoder, X_val, device=DEVICE)

            knn_acc, knn_f1 = knn_accuracy(train_emb, y_tr, val_emb, y_va, k=KNN_K)
            lp_acc, lp_f1   = linear_probe(train_emb, y_tr, val_emb, y_va)
            recalls         = retrieval_recall_at_k(val_emb, y_va, k_list=(1, 5))

            msg += (
                f" | kNN@{KNN_K} acc={knn_acc:.3f} f1={knn_f1:.3f}"
                f" | LP acc={lp_acc:.3f} f1={lp_f1:.3f}"
                f" | R@1={recalls['R@1']:.3f} R@5={recalls['R@5']:.3f}"
            )

        print(msg)

    package = {
        "feature_cols_original": cols,
        "cont_cols": cont_cols,
        "cat_cols": cat_cols,
        "label_col": LABEL_COL if has_labels else None,
        "scaler": scaler,
        "model_state_dict": model.state_dict(),
        "model_config": {
            "in_dim": X_train.shape[1],
            "hidden_dim": 256,
            "emb_dim": 128,
            "proj_dim": 128,
        },
        "train_config": {
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "temperature": TEMPERATURE,
            "val_split": VAL_SPLIT,
            "seed": SEED,
            "noise_std": NOISE_STD,
            "drop_prob": DROP_PROB,
            "scale_jitter": SCALE_JITTER,
            "mixup_alpha": MIXUP_ALPHA,
            "knn_k": KNN_K,
        },
    }

    with open(OUT_PKL, "wb") as f:
        pickle.dump(package, f)

    print(f"\n✅ Saved enhanced SimCLR package to: {OUT_PKL}")


if __name__ == "__main__":
    main()


Deduplicated rows: 20000 -> 15372  (removed 4628)


C:\Users\NIPUN\AppData\Local\Temp\ipykernel_21732\3085318866.py:316: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp_scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
C:\Users\NIPUN\AppData\Local\Temp\ipykernel_21732\3085318866.py:328: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Epoch 001/150 | loss=4.2814 | emb_std=0.0853
Epoch 002/150 | loss=3.8392 | emb_std=0.0849
Epoch 003/150 | loss=3.7530 | emb_std=0.0853
Epoch 004/150 | loss=3.7271 | emb_std=0.0855
Epoch 005/150 | loss=3.6886 | emb_std=0.0852
Epoch 006/150 | loss=3.6590 | emb_std=0.0855
Epoch 007/150 | loss=3.6249 | emb_std=0.0857
Epoch 008/150 | loss=3.6231 | emb_std=0.0854
Epoch 009/150 | loss=3.5954 | emb_std=0.0855
Epoch 010/150 | loss=3.5852 | emb_std=0.0856
Epoch 011/150 | loss=3.5791 | emb_std=0.0852
Epoch 012/150 | loss=3.5577 | emb_std=0.0854
Epoch 013/150 | loss=3.5419 | emb_std=0.0856
Epoch 014/150 | loss=3.5425 | emb_std=0.0852
Epoch 015/150 | loss=3.5338 | emb_std=0.0856
Epoch 016/150 | loss=3.5224 | emb_std=0.0855
Epoch 017/150 | loss=3.5138 | emb_std=0.0855
Epoch 018/150 | loss=3.5127 | emb_std=0.0855
Epoch 019/150 | loss=3.5373 | emb_std=0.0857
Epoch 020/150 | loss=3.5203 | emb_std=0.0858
Epoch 021/150 | loss=3.5001 | emb_std=0.0856
Epoch 022/150 | loss=3.4850 | emb_std=0.0852
Epoch 023/

In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import pickle

# =========================
# CONFIG
# =========================
CSV_PATH = "labeled_dataset.csv"
FEATURE_COLS = ["mjd", "fid", "magpsf", "sigmapsf", "ra", "dec", "isdiffpos"]
LABEL_COL = "transient_type"

BATCH_SIZE = 256
EPOCHS = 30
LR = 1e-3
TEMPERATURE = 0.2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(CSV_PATH)

X = df[FEATURE_COLS].apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

y = df[LABEL_COL].astype(str)

# =========================
# STANDARDIZE
# =========================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# =========================
# SIMCLR DATASET
# =========================
class SimCLRDataset(Dataset):
    def __init__(self, X, noise=0.03, drop=0.1):
        self.X = X.astype(np.float32)
        self.noise = noise
        self.drop = drop

    def augment(self, x):
        x = x + torch.randn_like(x) * self.noise
        mask = (torch.rand_like(x) > self.drop).float()
        return x * mask

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx])
        return self.augment(x), self.augment(x)


# =========================
# SIMCLR MODEL
# =========================
class Encoder(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )

    def forward(self, x):
        return self.net(x)


class SimCLR(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.encoder = Encoder(d)
        self.projector = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return h, F.normalize(z, dim=1)


def nt_xent(z1, z2, temp=0.2):
    N = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = torch.mm(z, z.T) / temp
    mask = torch.eye(2*N, device=z.device).bool()
    sim.masked_fill_(mask, -9e15)

    positives = torch.cat([torch.diag(sim, N), torch.diag(sim, -N)])
    loss = -positives + torch.logsumexp(sim, dim=1)
    return loss.mean()


# =========================
# TRAIN SIMCLR
# =========================
dataset = SimCLRDataset(X_scaled)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = SimCLR(X.shape[1]).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    losses = []
    for x1, x2 in loader:
        x1, x2 = x1.to(DEVICE), x2.to(DEVICE)
        _, z1 = model(x1)
        _, z2 = model(x2)

        loss = nt_xent(z1, z2)
        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {np.mean(losses):.4f}")

# =========================
# EMBEDDING EXTRACTION
# =========================
with torch.no_grad():
    H, _ = model(torch.tensor(X_scaled, dtype=torch.float32).to(DEVICE))
    H = H.cpu().numpy()

# =========================
# CLASSIFIER
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    H, y, test_size=0.2, random_state=42, stratify=y
)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)

print("\nClassifier accuracy:", clf.score(X_test, y_test))

# =========================
# SAVE EVERYTHING
# =========================
with open("simclr_classifier_pipeline.pkl", "wb") as f:
    pickle.dump({
        "scaler": scaler,
        "encoder_state": model.state_dict(),
        "feature_cols": FEATURE_COLS,
        "classifier": clf
    }, f)

print("✅ Saved: simclr_classifier_pipeline.pkl")


Epoch 1/30 | Loss: 3.6275
Epoch 2/30 | Loss: 2.9702
Epoch 3/30 | Loss: 2.9616
Epoch 4/30 | Loss: 2.4765
Epoch 5/30 | Loss: 2.2689
Epoch 6/30 | Loss: 2.3217
Epoch 7/30 | Loss: 2.2082
Epoch 8/30 | Loss: 2.0823
Epoch 9/30 | Loss: 2.1481
Epoch 10/30 | Loss: 2.2879
Epoch 11/30 | Loss: 2.0960
Epoch 12/30 | Loss: 2.1373
Epoch 13/30 | Loss: 2.0856
Epoch 14/30 | Loss: 2.1184
Epoch 15/30 | Loss: 1.9296
Epoch 16/30 | Loss: 1.9920
Epoch 17/30 | Loss: 1.9683
Epoch 18/30 | Loss: 2.1704
Epoch 19/30 | Loss: 1.7114
Epoch 20/30 | Loss: 1.9286
Epoch 21/30 | Loss: 1.9786
Epoch 22/30 | Loss: 1.8543
Epoch 23/30 | Loss: 1.9790
Epoch 24/30 | Loss: 1.7705
Epoch 25/30 | Loss: 1.9057
Epoch 26/30 | Loss: 1.8630
Epoch 27/30 | Loss: 1.7808
Epoch 28/30 | Loss: 1.9354
Epoch 29/30 | Loss: 2.1350
Epoch 30/30 | Loss: 1.9049

Classifier accuracy: 0.6363636363636364
✅ Saved: simclr_classifier_pipeline.pkl


In [8]:
##Classifier

import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- same SimCLR structure you trained with ---
class Encoder(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )
    def forward(self, x):
        return self.net(x)

class SimCLR(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.encoder = Encoder(d)
        self.projector = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return h, F.normalize(z, dim=1)


def predict_sample(sample_dict):
    # Load pipeline
    with open("simclr_classifier_pipeline.pkl", "rb") as f:
        pkg = pickle.load(f)

    feature_cols = pkg["feature_cols"]
    scaler = pkg["scaler"]
    clf = pkg["classifier"]

    # ✅ Load FULL SimCLR (because state_dict contains encoder+projector keys)
    simclr = SimCLR(len(feature_cols))
    simclr.load_state_dict(pkg["encoder_state"])   # (this is actually full simclr state dict)
    simclr.eval()

    # Build input in correct order
    x = np.array([[float(sample_dict[c]) for c in feature_cols]], dtype=np.float32)

    # Scale
    x_scaled = scaler.transform(x)

    # Get embedding
    with torch.no_grad():
        xt = torch.tensor(x_scaled, dtype=torch.float32)
        h, _ = simclr(xt)
        h_np = h.numpy()

    # Predict
    pred = clf.predict(h_np)[0]
    return pred


# Example
sample = {
    "mjd": 58278.40,
    "fid": 1,
    "magpsf": 16.69,
    "sigmapsf": 0.025,
    "ra": 307.79,
    "dec": 51.13,
    "isdiffpos": -1
}

print("Predicted class:", predict_sample(sample))


Predicted class: Supernova_Ia


C:\Users\NIPUN\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [ ]:
# === Classifier on embeddings & extended metrics (uses labeled_dataset.csv) ===
import os, pickle, math
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, log_loss, mean_absolute_error, mean_squared_error, r2_score, silhouette_score)
from sklearn.model_selection import train_test_split

CSV_PATH = 'labeled_dataset.csv'
LABEL_COL = 'transient_type'
PIPE_PKL = 'simclr_classifier_pipeline.pkl'
OUT_DIR = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)

# Load labels
df = pd.read_csv(CSV_PATH)
if LABEL_COL not in df.columns:
    raise ValueError(f"Label column '{LABEL_COL}' not found in {CSV_PATH}")
y = df[LABEL_COL].astype(str).values
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Obtain embeddings H_all: prefer existing variable H, else reconstruct from pipeline
try:
    H  # noqa: F821
    H_all = H
    print('Using in-memory embeddings variable H')
except Exception:
    if not os.path.exists(PIPE_PKL):
        raise RuntimeError('Embeddings not found in session and pipeline pkl missing. Run SimCLR training cell first or provide simclr_classifier_pipeline.pkl')
    with open(PIPE_PKL, 'rb') as f:
        pkg = pickle.load(f)
    feature_cols = pkg.get('feature_cols', None)
    scaler = pkg.get('scaler', None)
    encoder_state = pkg.get('encoder_state', None)
    if feature_cols is None:
        raise RuntimeError('feature_cols not present in pipeline pkl')
    X_raw = df[list(feature_cols)].apply(pd.to_numeric, errors='coerce')
    X_raw = X_raw.replace([np.inf, -np.inf], np.nan).fillna(X_raw.median())
    if scaler is not None:
        Xs = scaler.transform(X_raw.values)
    else:
        Xs = StandardScaler().fit_transform(X_raw.values)
    # Reconstruct encoder architecture (must match the one used when saving)
    import torch, torch.nn as nn, torch.nn.functional as F
    class Encoder(nn.Module):
        def __init__(self, d):
            super().__init__()
            self.net = nn.Sequential(nn.Linear(d, 256), nn.ReLU(), nn.Linear(256, 128))
        def forward(self, x):
            return self.net(x)
    class SimCLR(nn.Module):
        def __init__(self, d):
            super().__init__()
            self.encoder = Encoder(d)
            self.projector = nn.Sequential(nn.Linear(128,128), nn.ReLU(), nn.Linear(128,128))
        def forward(self, x):
            h = self.encoder(x)
            z = self.projector(h)
            return h, F.normalize(z, dim=1)
    simclr = SimCLR(Xs.shape[1])
    if encoder_state is None:
        raise RuntimeError('encoder_state not found in pipeline pkl')
    simclr.load_state_dict(encoder_state)
    simclr.eval()
    with torch.no_grad():
        Xt = torch.tensor(Xs, dtype=torch.float32)
        Ht, _ = simclr(Xt)
        H_all = Ht.numpy()
    print('Reconstructed embeddings from pipeline pkl')

# Train/test split
H_train, H_test, y_train, y_test = train_test_split(H_all, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

# Train logistic regression on embeddings
clf = LogisticRegression(max_iter=2000)
clf.fit(H_train, y_train)
y_pred = clf.predict(H_test)
proba = clf.predict_proba(H_test)

# Metrics
labels_sorted = np.unique(y_test)
acc = accuracy_score(y_test, y_pred)
prec_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
rec_macro  = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1_macro   = f1_score(y_test, y_pred, average='macro', zero_division=0)
prec_w = precision_score(y_test, y_pred, average='weighted', zero_division=0)
rec_w  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1_w   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print('Accuracy:', acc)
print('Macro Precision/Recall/F1:', prec_macro, rec_macro, f1_macro)
print('Weighted Precision/Recall/F1:', prec_w, rec_w, f1_w)

# Confusion matrix & report
cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)
print('
Confusion Matrix (rows=true, cols=pred):')
print(pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted))
print('
Classification Report:')
print(classification_report(y_test, y_pred, zero_division=0, target_names=le.inverse_transform(labels_sorted)))

# Per-class specificity / FPR / FNR
def per_class_rates(cm):
    n = cm.shape[0]
    spec, fpr, fnr = [], [], []
    for i in range(n):
        TP = cm[i,i]
        FP = cm[:,i].sum() - TP
        FN = cm[i,:].sum() - TP
        TN = cm.sum() - TP - FP - FN
        s = TN / (TN + FP) if (TN + FP) > 0 else 0.0
        fp = FP / (FP + TN) if (FP + TN) > 0 else 0.0
        fn = FN / (TP + FN) if (TP + FN) > 0 else 0.0
        spec.append(s); fpr.append(fp); fnr.append(fn)
    return spec, fpr, fnr
spec_list, fpr_list, fnr_list = per_class_rates(cm)
print('
Specificity per class:', spec_list)
print('Specificity (macro):', np.mean(spec_list))
print('FPR (macro):', np.mean(fpr_list))
print('FNR (macro):', np.mean(fnr_list))

# ROC-AUC (binary)
if len(labels_sorted) == 2:
    proba_pos = proba[:, list(clf.classes_).index(labels_sorted[1])]
    y_bin = (y_test == labels_sorted[1]).astype(int)
    try:
        auc = roc_auc_score(y_bin, proba_pos)
        print('ROC-AUC:', auc)
    except Exception as e:
        print('ROC-AUC could not be computed:', e)

# Log loss
try:
    ll = log_loss(y_test, proba)
    print('Log Loss:', ll)
except Exception as e:
    print('Log loss could not be computed:', e)

# Regression-style metrics on integer labels vs predicted labels
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = math.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print('MAE:', mae, 'MSE:', mse, 'RMSE:', rmse, 'R2:', r2)

# Silhouette on embeddings (if possible)
try:
    sil = silhouette_score(H_test, y_test)
    print('Silhouette score (embeddings, test set):', sil)
except Exception as e:
    print('Silhouette score could not be computed:', e)

# Save classifier and label encoder
out_path = os.path.join(OUT_DIR, 'classifier_embeddings.pkl')
with open(out_path, 'wb') as f:
    pickle.dump({'clf': clf, 'label_encoder': le, 'classes': clf.classes_}, f)
print('Saved classifier to:', out_path)